In [ ]:
import bw2data, bw2io
import bw2calc
import os
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import numpy as np
import pandas as pd
import xarray as xr
plt.rcParams['font.family'] = 'Times New Roman'
matplotlib.style.use('tableau-colorblind10')

### read in the two sets of dpGWP prepared: previous default dpGWP one same folder here as `GWP_output_allyears.xlsx`, then a new set of dpGWP w/h fixed static AGWP[CO2] under `../AGWPCO2_fixed_IPCCAR6/dpGWP_CO2CH4N2O_output_fixed_AGWPCO2.xlsx`

In [ ]:
file1 = '../AGWPCO2_fixed_IPCCAR6/dpGWP_CO2CH4N2O_output_fixed_AGWPCO2.xlsx'   
file2 = 'GWP_output_allyears.xlsx'   
df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)

In [ ]:
### some clean up needed 
df1_clean = df1.copy()
df1_clean['SSP'] = df1_clean['SSP'].ffill()
df1_clean['ModelYear'] = df1_clean['ModelYear'].ffill()

df2_clean = df2.loc[:, ~df2.columns.str.startswith('Unnamed')].copy()
df2_clean['SSP'] = df2_clean['SSP'].ffill()
df2_clean['ModelYear'] = df2_clean['ModelYear'].ffill()


In [ ]:
df1_clean.head()

In [ ]:
df2_clean.head()

### compare two appraoch of pGWP100

In [ ]:
df1_clean[df1_clean['Year'] == 100]

In [ ]:
df2_clean[df2_clean['Year'] == 100]

### plot by MY/SSP and compare the two sets of dpGWP::  now adding GWP[CO2]

In [ ]:
## CH4
fig, axs = plt.subplots(3, 3, figsize=(12, 8), sharex=True)
fig.subplots_adjust(hspace=0.35, wspace=0.25)

model_years = [2030, 2040, 2050]
ssp_map = {'ssp119': '1-19', 'ssp245': '2-45', 'ssp585': '5-85'}
colors = {'1-19': '#006BA4', '2-45': '#FF800E', '5-85': 'gray'}
#plt.rcParams['axes.prop_cycle'].by_key()['color'] blue: #006BA4  orange: #FF800E

for i, my in enumerate(model_years):
    # Compute per-row max Y for dynamic scaling
    max_y = 0
    for ssp1, ssp2 in ssp_map.items():
        y1_max = df1_clean.query("SSP == @ssp1 and ModelYear == @my and Year != 0")["CH4_GWP"].max()
        y2_max = df2_clean.query("SSP == @ssp2 and ModelYear == @my and Year != 0")["CH4_GWP"].max()
        max_y = max(max_y, y1_max, y2_max)
    y_limit = round((max_y + 10) / 10) * 10  # round up to nearest 10

    for j, (ssp1, ssp2) in enumerate(ssp_map.items()):
        ax = axs[i, j]

        # filter for current MY/SSP, excluding Year == 0
        y1 = df1_clean.query("SSP == @ssp1 and ModelYear == @my and Year != 0")
        y2 = df2_clean.query("SSP == @ssp2 and ModelYear == @my and Year != 0")

        # Set 2 GWP (solid)
        ax.plot(
            y2["Year"], y2["CH4_GWP"],
            label='dpAGWP[both CH4 and CO2]', color=colors[ssp2], linewidth=2
        )

        # Set 1 GWP (dashed)
        ax.plot(
            y1["Year"], y1["CH4_GWP"],
            label='dpAGWP[CH4], fixed AGWP[CO2]', linestyle='--', color=colors[ssp2], linewidth=1.5
        )

        ax.set_ylim(20, y_limit)
        ax.set_yticks(np.linspace(20, y_limit, 6))

        if j == 0:
            ax.set_ylabel(f'MY = {my}\ndpGWP1-100[CH$_4$]')
        if i == 2:
            ax.set_xlabel(f'SSP {ssp2}')
        if i == 0 and j == 0:
            ax.legend()
        if j > 0:
            ax.set_ylabel(None)
            ax.tick_params(labelleft=False)  


plt.suptitle(r'dpGWP[CH$_4$]: dpAGWP[CH$_4$] AND fixed AGWP[CO$_2$] (dashed) vs. dpAGWP[CH$_4$] AND dpAGWP[CO$_2$] (solid)', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('output_plot/new0707/GWP1-100_CH4_Compare_dpGWP.jpg', dpi=300)
plt.show()


In [ ]:
## CO2
fig, axs = plt.subplots(3, 3, figsize=(12, 8), sharex=True)
fig.subplots_adjust(hspace=0.35, wspace=0.25)

model_years = [2030, 2040, 2050]
ssp_map = {'ssp119': '1-19', 'ssp245': '2-45', 'ssp585': '5-85'}
colors = {'1-19': '#006BA4', '2-45': '#FF800E', '5-85': 'gray'}
#plt.rcParams['axes.prop_cycle'].by_key()['color'] blue: #006BA4  orange: #FF800E

for i, my in enumerate(model_years):
    # Compute per-row max Y for dynamic scaling
    #max_y = 0
    #for ssp1, ssp2 in ssp_map.items():
    #    y1_max = df1_clean.query("SSP == @ssp1 and ModelYear == @my and Year != 0")["CO2_GWP"].max()
    #    y2_max = df2_clean.query("SSP == @ssp2 and ModelYear == @my and Year != 0")["CO2_GWP"].max()
    #    max_y = max(max_y, y1_max, y2_max)
    y_limit = 2   # diff scale from CH4 

    for j, (ssp1, ssp2) in enumerate(ssp_map.items()):
        ax = axs[i, j]

        # filter for current MY/SSP, excluding Year == 0
        y1 = df1_clean.query("SSP == @ssp1 and ModelYear == @my and Year != 0")
        y2 = df2_clean.query("SSP == @ssp2 and ModelYear == @my and Year != 0")

        # Set 2 GWP (solid)
        ax.plot(
            y2["Year"], y2["CO2_GWP"],
            label='dpAGWP[both CO2 and CO2]', color=colors[ssp2], linewidth=2
        )

        # Set 1 GWP (dashed)
        ax.plot(
            y1["Year"], y1["CO2_GWP"],
            label='dpAGWP[CO2], fixed AGWP[CO2]', linestyle='--', color=colors[ssp2], linewidth=1.5
        )

        ##### diff scale for CO2 
        ax.set_ylim(0.8, 1.4)
        ax.set_yticks(np.arange(0.8, 1.5, 0.2))  #ax.set_yticks(np.linspace(0, 2, 0.5))

        if j == 0:
            ax.set_ylabel(f'MY = {my}\ndpGWP1-100[CO$_2$]')
        if i == 2:
            ax.set_xlabel(f'SSP {ssp2}')
        if i == 0 and j == 0:
            ax.legend()
        if j > 0:
            ax.set_ylabel(None)
            ax.tick_params(labelleft=False)  


plt.suptitle(r'dpGWP[CO$_2$]: dpAGWP[CO$_2$] AND fixed AGWP[CO$_2$] (dashed) vs. dpAGWP[CO$_2$] AND dpAGWP[CO$_2$] (solid)', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('output_plot/new0707/GWP1-100_CO2_Compare_dpGWP.jpg', dpi=300)
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(12, 8), sharex=True)
fig.subplots_adjust(hspace=0.35, wspace=0.25)

model_years = [2030, 2040, 2050]
ssp_map = {'ssp119': '1-19', 'ssp245': '2-45', 'ssp585': '5-85'}
colors = {'1-19': '#006BA4', '2-45': '#FF800E', '5-85': 'gray'}
#plt.rcParams['axes.prop_cycle'].by_key()['color'] blue: #006BA4  orange: #FF800E

for i, my in enumerate(model_years):
    # Compute per-row max Y for dynamic scaling
    max_y = 0
    for ssp1, ssp2 in ssp_map.items():
        y1_max = df1_clean.query("SSP == @ssp1 and ModelYear == @my and Year != 0")["N2O_GWP"].max()
        y2_max = df2_clean.query("SSP == @ssp2 and ModelYear == @my and Year != 0")["N2O_GWP"].max()
        max_y = max(max_y, y1_max, y2_max)
    y_limit = round((max_y + 10) / 10) * 10  # round up to nearest 10
    
    for j, (ssp1, ssp2) in enumerate(ssp_map.items()):
        ax = axs[i, j]

        # filter for current MY/SSP, excluding Year == 0
        y1 = df1_clean.query("SSP == @ssp1 and ModelYear == @my and Year != 0")
        y2 = df2_clean.query("SSP == @ssp2 and ModelYear == @my and Year != 0")

        # Set 2 GWP (solid)
        ax.plot(
            y2["Year"], y2["N2O_GWP"],
            label='dpAGWP[both N2O and CO2]', color=colors[ssp2], linewidth=2
        )

        # Set 1 GWP (dashed)
        ax.plot(
            y1["Year"], y1["N2O_GWP"],
            label='dpAGWP[N2O], fixedAGWP[CO2]', linestyle='--', color=colors[ssp2], linewidth=1.5
        )

        # hard code smallers y
        ax.set_ylim(140, y_limit)
        ax.set_yticks(np.linspace(140, y_limit, 6))

        if j == 0:
            ax.set_ylabel(f'MY = {my}\ndpGWP1-100[N$_2$O]')
        if i == 2:
            ax.set_xlabel(f'SSP {ssp2}')
        if i == 0 and j == 0:
            ax.legend()
        if j > 0:
            ax.set_ylabel(None)
            ax.tick_params(labelleft=False)  


plt.suptitle(r'dpGWP[N$_2$O]: dpAGWP[N$_2$O] AND fixed AGWP[CO$_2$] (dashed) vs. dpAGWP[N$_2$O] AND dpAGWP[CO$_2$] (solid)', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('output_plot/new0707/GWP1-100_N2O_Compare_dpGWP.jpg', dpi=300)
plt.show()

### just GWP20 & 100, now adding CO2, new layout 2*3 

In [ ]:
# fix the col naming: DF1 SSP codes (e.g., 'ssp119') to DF2 style (e.g., '1-19')
ssp_map = {'ssp119': '1-19', 'ssp245': '2-45', 'ssp585': '5-85'}
df1_clean["SSP"] = df1_clean["SSP"].replace(ssp_map)

# same name GWP20 and GWP100 after SSP rename, only MY = 2030/40/50
def extract_pgwp(df, gas, horizon, model_years=[2030, 2040, 2050]):
    return (
        df[
            (df['Year'] == horizon) &
            (df['ModelYear'].isin(model_years))
        ][['SSP', 'ModelYear', f'{gas}_GWP']]
        .rename(columns={f'{gas}_GWP': f'GWP{horizon}'})
    )

#  CH4 and N2O data from both GWP sets
ch4_20_s1 = extract_pgwp(df1_clean, 'CH4', 20)
ch4_100_s1 = extract_pgwp(df1_clean, 'CH4', 100)
ch4_20_s2 = extract_pgwp(df2_clean, 'CH4', 20)
ch4_100_s2 = extract_pgwp(df2_clean, 'CH4', 100)

n2o_20_s1 = extract_pgwp(df1_clean, 'N2O', 20)
n2o_100_s1 = extract_pgwp(df1_clean, 'N2O', 100)
n2o_20_s2 = extract_pgwp(df2_clean, 'N2O', 20)
n2o_100_s2 = extract_pgwp(df2_clean, 'N2O', 100)

### adding CO2, only GWP100 needed, as GWP20 == GWP100 for both approach: 
co2_20_s1 = extract_pgwp(df1_clean, 'CO2', 20)
co2_100_s1 = extract_pgwp(df1_clean, 'CO2', 100)
co2_20_s2 = extract_pgwp(df2_clean, 'CO2', 20)
co2_100_s2 = extract_pgwp(df2_clean, 'CO2', 100)


In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(6, 9), sharex=True)
axs = axs.reshape(3, 2)
colors = {'1-19': '#006BA4', '2-45': '#FF800E', '5-85': 'gray'}
# --- CO2 (shared y) ---
for label, group2 in co2_20_s2.groupby("SSP"):
    group1 = co2_20_s1[co2_20_s1["SSP"] == label]
    axs[0, 0].plot(group2['ModelYear'], group2['GWP20'], color=colors[label], marker='o', linewidth=2)
    axs[0, 0].plot(group1['ModelYear'], group1['GWP20'], color=colors[label], linestyle='--', marker='o', linewidth=1.5)

for label, group2 in co2_100_s2.groupby("SSP"):
    group1 = co2_100_s1[co2_100_s1["SSP"] == label]
    axs[0, 1].plot(group2['ModelYear'], group2['GWP100'], color=colors[label], marker='o', linewidth=2)
    axs[0, 1].plot(group1['ModelYear'], group1['GWP100'], color=colors[label], linestyle='--', marker='o', linewidth=1.5)
axs[0, 0].set_ylim(0.8, 1.2)
axs[0, 1].set_ylim(0.8, 1.2)


# --- CH4 ---
for label, group2 in ch4_20_s2.groupby("SSP"):
    group1 = ch4_20_s1[ch4_20_s1["SSP"] == label]
    axs[1, 0].plot(group2['ModelYear'], group2['GWP20'], color=colors[label], marker='o', linewidth=2)
    axs[1, 0].plot(group1['ModelYear'], group1['GWP20'], color=colors[label], linestyle='--', marker='o', linewidth=1.5)

for label, group2 in ch4_100_s2.groupby("SSP"):
    group1 = ch4_100_s1[ch4_100_s1["SSP"] == label]
    axs[1, 1].plot(group2['ModelYear'], group2['GWP100'], color=colors[label], marker='o', linewidth=2)
    axs[1, 1].plot(group1['ModelYear'], group1['GWP100'], color=colors[label], linestyle='--', marker='o', linewidth=1.5)

# --- N2O ---
for label, group2 in n2o_20_s2.groupby("SSP"):
    group1 = n2o_20_s1[n2o_20_s1["SSP"] == label]
    axs[2, 0].plot(group2['ModelYear'], group2['GWP20'], color=colors[label], marker='o', linewidth=2)
    axs[2, 0].plot(group1['ModelYear'], group1['GWP20'], color=colors[label], linestyle='--', marker='o', linewidth=1.5)

for label, group2 in n2o_100_s2.groupby("SSP"):
    group1 = n2o_100_s1[n2o_100_s1["SSP"] == label]
    axs[2, 1].plot(group2['ModelYear'], group2['GWP100'], color=colors[label], marker='o', linewidth=2)
    axs[2, 1].plot(group1['ModelYear'], group1['GWP100'], color=colors[label], linestyle='--', marker='o', linewidth=1.5)

titles = [
    r'CO$_2$ pGWP20', r'CO$_2$ pGWP100',
    r'CH$_4$ pGWP20', r'CH$_4$ pGWP100',
    r'N$_2$O pGWP20', r'N$_2$O pGWP100',
]
for ax, title in zip(axs.flat, titles):
    ax.set_title(title, fontsize=12)
    ax.set_xticks([2030, 2040, 2050])
    ax.set_xlabel("ModelYear")

# --- legend and labels ---
handles = []  # store Line2D objects for legend
for label, group2 in co2_20_s2.groupby("SSP"):
    group1 = co2_20_s1[co2_20_s1["SSP"] == label]
    line1, = axs[0, 0].plot(group2['ModelYear'], group2['GWP20'], color=colors[label], marker='o', linewidth=2, label=label)
    axs[0, 0].plot(group1['ModelYear'], group1['GWP20'], color=colors[label], linestyle='--', marker='o', linewidth=1.5)
    handles.append(line1)  # collect just once for each SSP
axs[0, 0].legend(handles=handles, title="SSP", fontsize=10)

axs[0, 0].set_ylabel("pGWP")
axs[1, 0].set_ylabel("pGWP")
axs[2, 0].set_ylabel("pGWP")


plt.suptitle(r'pGWP20 & pGWP100 for CO$_2$, CH$_4$, and N$_2$O'
             '\n - solid: dpAGWP[CO$_2$/CH$_4$/N$_2$O] & dpAGWP[CO$_2$]'
             '\n - dashed: dpAGWP[CO$_2$/CH$_4$/N$_2$O] & fixed AGWP[CO$_2$]',
             fontsize=12)
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.savefig('output_plot/new0707/GWP20-100_CO2_N2O_CH4_compare_twosets_dpGWP_grid.jpg', dpi=300)
plt.show()
